# This Notebook Processes Example Questions, Answers and Judgements of selected Models

In [1]:
import pandas as pd
import numpy as np
import re
import json
import os

from itertools import islice

In [2]:
bench_name = 'arena-hard-v0.1'

judge_model = 'gpt-4o'
baseline_model = 'gpt-4-0314'

model_list = ['gpt-4o-mini', 'mistral_large_2', 'mistral_nemo', 'mistral_nemo_fp8', 'mistral_nemo_int4', 'llama3_1_405b', 'llama3_1_70b', 'llama3_1_70b_fp8_dyn', 'llama3_1_70b_awq_int4', 'llama3_1_8b', 'llama3_1_8b_fp8', 'llama3_1_8b_int4', 'llama3_70b']
guided_model_list = [model_name + '_guided' for model_name in model_list]

question_file = f"arena-hard-auto/data/{bench_name}/question.jsonl"
guidance_file = f"arena-hard-auto/data/{bench_name}/guidance.jsonl"


if os.path.exists(question_file):
    print(f"Found question file at '{question_file}'")

if os.path.exists(guidance_file):
    print(f"Found guidance file at '{guidance_file}'")

Found question file at 'arena-hard-auto/data/arena-hard-v0.1/question.jsonl'
Found guidance file at 'arena-hard-auto/data/arena-hard-v0.1/guidance.jsonl'


In [3]:
def get_questions(question_file: str):
    """Load questions from a file into a list."""

    questions = {}
    with open(question_file, "r") as ques_file:
        for line in ques_file:
            if line:
                line = json.loads(line)
                questions[line["question_id"]] = line["turns"][0]["content"]
    return questions

In [4]:
def get_ques_clusters(question_file: str):
    """Load questions from a file into a list."""

    clusters = {}
    with open(question_file, "r") as ques_file:
        for line in ques_file:
            if line:
                line = json.loads(line)
                clusters[line["question_id"]] = line["cluster"]
    return clusters

In [5]:
def get_answers(answer_file: str):
    """Load model answers."""

    answers = {}
    with open(answer_file, "r") as ans_file:
        for line in ans_file:
            if line:
                line = json.loads(line)
                answers[line["question_id"]] = line["choices"][0]["turns"][0]["content"]
    return answers

In [6]:
def load_guidance(guidance_file: str):
    """Load guidances from a file."""
    guidances = {}
    with open(guidance_file, "r") as fin:
        for line in fin:
            if line:
                line = json.loads(line)
                guidances[line["question_id"]] = line["guidance"]
    return guidances

In [7]:
all_questions = get_questions(question_file)
all_guidances = load_guidance(guidance_file)
all_clusters = get_ques_clusters(question_file)

all_answers = []
all_guided_answers = []

for model_name in model_list:
    answer_dir_name = f'arena-hard-auto/data/{bench_name}/model_answer/{model_name}.jsonl'
    if not os.path.exists(answer_dir_name):
        print(f"No Bench results file found at '{answer_dir_name}'")
    else: 
        #print(f"Found Bench results file at '{answer_dir_name}'")
        all_answers.append(get_answers(answer_dir_name))

for model_name in guided_model_list:
    answer_dir_name = f'arena-hard-auto/data/{bench_name}/model_answer/{model_name}.jsonl'
    if not os.path.exists(answer_dir_name):
        print(f"No Bench results file found at '{answer_dir_name}'")
    else: 
        #print(f"Found Bench results file at '{answer_dir_name}'")
        all_guided_answers.append(get_answers(answer_dir_name))

In [8]:
selected_questions = list(islice(all_questions.items(), 30))
print(selected_questions[0])

('328c149ed45a41c0b9d6f14659e63599', 'Use ABC notation to write a melody in the style of a folk tune.')


In [9]:
selected_quidances = {qid: all_guidances[qid] for qid, _ in selected_questions}
print(selected_quidances[selected_questions[0][0]])

### Contextual Information
ABC notation is a shorthand form of musical notation that uses the letters A through G to represent notes, along with other symbols to indicate rhythm, key, and other musical elements. It is particularly useful for notating folk tunes and other simple melodies.

### Hints and Tips
- **Key Signature**: Choose a key signature that is common in folk music, such as G major or D major.
- **Time Signature**: Folk tunes often use simple time signatures like 4/4 or 3/4.
- **Note Lengths**: Use a default note length that suits the style of the melody, such as eighth notes (L:1/8).
- **Melodic Structure**: Folk tunes typically have a repetitive and easy-to-follow structure. Think about common folk motifs and patterns.
- **Chords**: Consider adding chords to enhance the melody. Use quotation marks around chord names.

### Step-by-Step Plan
1. **Set Up the Header**:
   - Start with the reference number (`X:1`).
   - Add a title for your tune (`T:Your Folk Tune Title`).
 

In [10]:
selected_clusters = {qid: all_clusters[qid] for qid, _ in selected_questions}
print(selected_clusters[selected_questions[0][0]])

ABC Sequence Puzzles & Groups


In [11]:
selected_answers = {model_name: {qid: all_answers[i][qid] for qid, _ in selected_questions} for i, model_name in enumerate(model_list)}

print(10*"= " + model_list[0] + 10*" =" + "\n")
print(selected_answers[model_list[0]][selected_questions[0][0]])

= = = = = = = = = = gpt-4o-mini = = = = = = = = = =

Sure! Here’s a simple melody written in ABC notation that captures the essence of a folk tune:

```abc
X: 1
T: Folk Tune
M: 4/4
L: 1/8
K: C
|: G2 E2 C2 D2 | E4 D2 C2 | G2 E2 C2 D2 | E4 D4 :|
|: C2 E2 G2 A2 | B4 A2 G2 | C2 E2 G2 A2 | B4 A4 :|
```

### Explanation:
- **X:** Reference number of the tune.
- **T:** Title of the tune.
- **M:** Meter (4/4 time).
- **L:** Default note length (1/8).
- **K:** Key signature (C major).
- The melody consists of two sections, each repeated (indicated by the colons).

You can use an ABC notation software or online converter to play this melody and hear how it sounds! Enjoy your folk tune!


In [12]:
selected_guided_answers = {model_name: {qid: all_guided_answers[i][qid] for qid, _ in selected_questions} for i, model_name in enumerate(guided_model_list)}

print(10*"= " + guided_model_list[0] + 10*" =" + "\n")
print(selected_guided_answers[guided_model_list[0]][selected_questions[0][0]])

= = = = = = = = = = gpt-4o-mini_guided = = = = = = = = = =

Here’s a simple folk tune written in ABC notation:

```
X:1
T:Simple Folk Tune
M:4/4
L:1/8
K:G
"G"G2 D2 | "C" E2 G2 | "G" A2 B2 | "D" A2 G2 |
"G"G2 D2 | "C" E2 G2 | "G" A2 B2 | "D" G4 ||
"G" B2 A2 | "C" E2 D2 | "G" G2 A2 | "D" A4 |
"G" B2 A2 | "C" E2 D2 | "G" G2 A2 | "D" G4 ||
```

### Explanation:
1. **Header Information**:
   - `X:1` indicates the reference number.
   - `T:Simple Folk Tune` is the title of the tune.
   - `M:4/4` specifies the time signature.
   - `L:1/8` sets the default note length to eighth notes.
   - `K:G` indicates the key signature of G major.

2. **Melody Composition**:
   - The melody uses simple phrases and repeats sections to create a folk-like quality.
   - Chords are added before the notes to enhance the harmonic structure.

3. **Structure**:
   - The tune consists of two sections, each repeated, which is common in folk music.

Feel free to play around with the notes and chords to create variatio

In [13]:
data = []

for qid, question in selected_questions:
    system_prompt = f"""
            You are a sophisticated AI-Expert there to help users solve tasks in several domains efficiently and accurately.
            Now solve the following task from the domain "{selected_clusters[qid]}".\n
            """
    
    guided_system_prompt = f"""
            You are a sophisticated AI assistant with the task to solve a question in the domain of {selected_clusters[qid]}. 
            Here is important guidance to solve the task that will be given to you:\n{selected_quidances[qid]}\n\n
            Your task is:\n"""

    row = {"qid": qid, "question": question, "cluster": selected_clusters[qid], "guidance": selected_quidances[qid], "system_prompt": system_prompt, "guided_system_prompt": guided_system_prompt}
    for model_name in model_list:
        row[model_name] = selected_answers[model_name][qid]
    for model_name in guided_model_list:
        row[model_name] = selected_guided_answers[model_name][qid]
    data.append(row)

df = pd.DataFrame(data)
df


,qid,question,cluster,guidance,system_prompt,guided_system_prompt,gpt-4o-mini,mistral_large_2,mistral_nemo,mistral_nemo_fp8,...,mistral_nemo_fp8_guided,mistral_nemo_int4_guided,llama3_1_405b_guided,llama3_1_70b_guided,llama3_1_70b_fp8_dyn_guided,llama3_1_70b_awq_int4_guided,llama3_1_8b_guided,llama3_1_8b_fp8_guided,llama3_1_8b_int4_guided,llama3_70b_guided
0,328c149ed45a41c0b9d6f14659e63599,Use ABC notation to write a melody in the styl...,ABC Sequence Puzzles & Groups,### Contextual Information\nABC notation is a ...,\n You are a sophisticated AI-Exper...,\n You are a sophisticated AI assis...,Sure! Here’s a simple melody written in ABC no...,"Sure, I'd be happy to help with that! ABC nota...",Here's a simple folk tune melody using ABC not...,"Sure, here's a simple folk tune melody written...",...,"Based on the provided guidance, here's a simpl...","Based on the provided guidance, here's a simpl...",To write a melody in the style of a folk tune ...,X:1\nT:Folk Tune in G Major\nM:4/4\nL:1/8\nK:G...,X:1\nT:Folk Tune in G Major\nM:4/4\nL:1/8\nK:G...,Here's a simple folk tune in ABC notation:\n\n...,### Contextual Information\n\nABC notation is ...,Let's create a simple folk tune using ABC nota...,Let's create a simple folk tune using ABC nota...,Here's a folk-inspired melody in ABC notation:...
1,b43c07656ead4150b360294ee932b410,SOLVE THIS IN C++ : There are three cards with...,ABC Sequence Puzzles & Groups,### Contextual Information\nYou are given a pr...,\n You are a sophisticated AI-Exper...,\n You are a sophisticated AI assis...,To solve the problem of determining whether we...,"To solve this problem, we need to determine if...",Here's a C++ solution for the given problem. T...,Here's a C++ solution for the given task. The ...,...,Here's a C++ solution for the given problem. T...,Here's a C++ solution for the given problem. T...,Here is a simple C++ solution for the problem:...,Here is a C++ solution for the problem:\n\n```...,Here is a C++ solution for the problem:\n\n```...,Here is a C++ solution for the problem:\n\n```...,Here is a C++ solution for the problem:\n\n```...,Here is the C++ code to solve the problem:\n\n...,Here is the C++ code to solve the problem:\n\n...,Here is the C++ solution for the problem:\n```...
2,1f07cf6d146d4038b2b93aaba3935ce0,Explain the book the Alignment problem by Bria...,AI & Sequence Alignment Challenges,"### Contextual Information\n\n""The Alignment P...",\n You are a sophisticated AI-Exper...,\n You are a sophisticated AI assis...,**The Alignment Problem: Machine Learning and ...,"### Synopsis of ""The Alignment Problem"" by Bri...","**Book Synopsis: ""The Alignment Problem: Machi...","**Book Synopsis: ""The Alignment Problem"" by Br...",...,"**Synopsis of ""The Alignment Problem"" by Brian...","**Synopsis of ""The Alignment Problem"" by Brian...",**The Alignment Problem by Brian Christian: A ...,"**Synopsis of ""The Alignment Problem"" by Brian...","**Synopsis of ""The Alignment Problem"" by Brian...","""The Alignment Problem"" by Brian Christian is ...",**The Alignment Problem** by Brian Christian i...,"**Book Synopsis:**\n\n""The Alignment Problem"" ...","**Book Synopsis:**\n\n""The Alignment Problem"" ...","**Synopsis of ""The Alignment Problem"" by Brian..."
3,9f25ff7c0d6a4d74846bfe76af8d925c,Design a semikinematic mounting for a right an...,AI & Sequence Alignment Challenges,### Contextual Information\nDesigning a semiki...,\n You are a sophisticated AI-Exper...,\n You are a sophisticated AI assis...,Designing a semikinematic mounting for a right...,Designing a semikinematic mounting for a right...,To design a semikinematic mounting for a right...,To design a semikinematic mounting for a right...,...,"Based on the provided guidance, here's a step-...","Based on the provided guidance, here's a detai...",To design a semikinematic mounting for a right...,To design a semikinematic mounting for a right...,To design a semikinematic mounting for a right...,To design a semikinematic mountin

In [16]:
print(df['llama3_1_405b_guided'][5])

**Detecting and Removing 1-Pixel Width White Horizontal Lines from Black and White Images**

To efficiently detect and remove 1-pixel width white horizontal lines from black and white images, you can utilize Python and the OpenCV library. Below is a step-by-step guide to accomplish this task:

### Step 1: Install and Import Libraries

First, ensure you have OpenCV installed. You can install it via pip:

```bash
pip install opencv-python
```

Next, import the necessary libraries:

```python
import cv2
import numpy as np
```

### Step 2: Load the Image

Load your black and white image in grayscale mode using OpenCV:

```python
image = cv2.imread('your_image_path.jpg', cv2.IMREAD_GRAYSCALE)
```

### Step 3: Detect Horizontal Lines

Define a horizontal kernel suitable for detecting 1-pixel wide lines. Then, apply morphological operations to highlight these lines:

```python
# Define a horizontal kernel
kernel = np.ones((1, 5), np.uint8)

# Apply morphological operations to detect horizonta

In [17]:
# Define the relative path to the target directory
save_path = '../results/data/bench_qa_deepdive.parquet'

# Save the DataFrame to the Parquet file
df.to_parquet(save_path, index=False)